# NINA Output CSV for Spectra
## Adjust Celestial Coordinates for Assigned targets


In [ ]:
!mkdir -p data_folder

# INITIALIZE CELESTIAL COORDINATES PROCESS

In [ ]:
#pip install --upgrade astropy astroquery googlesearch-python wikipedia pandas


In [ ]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.io import ascii
import numpy as np
import pandas as pd
# importing the module
import wikipedia as wiki
from IPython.display import Markdown as md

In [ ]:
# ra_dec_offset_v5 and output files naming
celestial_coordinates_version = "v1"
data_folder_name = "data_folder"
output_csvfilename = f"celestial_coordinates_version_{celestial_coordinates_version}.csv"
print(f"celestial_coordinates_version is: {celestial_coordinates_version}")
print(f"data_folder_name is: {data_folder_name}")
print(f"output_csvfilename is: {output_csvfilename}")

In [ ]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u

obs_tel = "BARO"
obs_loc = "San Diego"
obs_lat = 32.6 * u.deg  # for san diego
obs_lon = -116.3 * u.deg # for san diego
obs_hgt = 1131 * u.m # for BARO
safe_lim = 10 * u.deg # Account for BARO Telescop stops 
max_mag = 8 # max star magnitudes to consider
min_ra = 12 # min ra limit for BARO
max_ra = 18 # max ra limit for BARO

print(f"Observers Location is: {obs_loc}")
print(f"Observers Telescope is: {obs_tel}")
print(f"Observers Lattitude is: {obs_lat}")
print(f"Observers Longitude is: {obs_lon}")
print(f"Observers Height is: {obs_hgt}")
print(f"Safe Limit for {obs_tel} is: {safe_lim}")
print(f"Max Mag to Query is: {max_mag}")
print(f"Min RA to Query is: {min_ra}")
print(f"Max RA to Query is: {max_ra}")

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude:.4f}")
print(f"Azimuth: {azimuth:.4f}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")

# Set global offset for spectra in frame to include zero-order
global_offset_arcmin = 2 # reduced from 3.5
print(f"Global offset RA & Dec arcmin by {global_offset_arcmin} * sin(camera rotation angle)")

In [ ]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith at {obs_loc} is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits at {obs_tel} are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

In [ ]:
# set target default name
target_default_name = "HD"

In [ ]:
def compute_exposure_time(mag: float) -> float:
    """
    Compute exposure time (in seconds) to reach 50,000 flux
    given the apparent magnitude, using the refit model
    (excluding La Superba).
    """
    a = 0.9325
    b = 1.0569
    c = -12.325
    target_flux = 50000

    log_flux = np.log(target_flux)
    log_exp = (log_flux + a * mag + c) / b
    return np.exp(log_exp)*2.5



In [ ]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    #original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    print(f'\nsky_PA: {sky_PA}')
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    
    print(f'\noffset_dist: {offset_dist}')
    
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)
    
    print(f'\ndx: {dx} dy: {dy}')
          

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)
    
    print(f'\noffset_frame = {offset_frame}')

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    print(f'\noffset_coord = {offset_coord}')
    print(f'\nnew_coord_ = {new_coord_}')
    
    return new_coord_



In [ ]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
#query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

#results = make_api_request(query)
#if results:
#    print("Data received:", results)


In [ ]:
def googlesearchurl(tname):
    try:
        from googlesearch import search
    except ImportError:
        print("No module named 'google' found")
    #
    # to search
    query = f'{tname} Wikipedia Astronomy'
    
    for j in search(query, tld="co.us", num=10, stop=10, pause=2):
            print(j)
    #for url in search(query, num_results=10, lang="en", unique=True):
    #    print(url)
#
# example
#
#tname = "RR Lyrae"
#googlesearchurl(tname)

In [ ]:
def wikisearchurl(tname):
    # Set language to English (optional)
    wiki.set_lang("en")
    
    # Search for a topic
    results = wiki.search(tname)
    print("Search Results:", results)
    
    # Get a summary of the first result
    summary = wiki.summary(results[0], sentences=1)
    print("\nSummary:", summary)
    
    # Get the full page object
    page = wiki.page(results[0])
    print("\nPage Title:", page.title)
    print("Page URL:", page.url)
#
# example
#
#tname = "RR Lyrae"
#wikisearchurl(tname)

In [ ]:
def obtain_info_for_star(tname):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    original_coord = SkyCoord.from_name(tname)
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord)

In [ ]:
def obtain_info_for_adhoc_star(tname, ara, adec):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(tname)
    original_coord = SkyCoord(ara, adec, frame='icrs', unit='deg')
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord)

In [ ]:
def obtain_adjcoord_for_planet(pname, pcoord):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(target_name)
    
    print(pcoord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(pcoord.ra, pcoord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {target_name} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {pcoord.ra.hour:.4f}, {pcoord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {pcoord.ra.deg:.4f}, {pcoord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(pcoord.ra.deg-new_coord.ra.deg):.4f}, \
          {(pcoord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");googlesearchurl(target_name);print("\n")

    return(new_coord)

In [ ]:
# ---  CREATE A DATAFRAME
column_names = ["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp"] 
df = pd.DataFrame(columns=column_names)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Or a large integer like 9999
ridx = 0

# CELESTIAL COORDINATES FOR NEW STAR 3C273

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "3C273"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 12.9;  target_alt_name = "Quasar_3C273" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# OUTPUT CONSOLIDATED CSV FILE FOR BARO

In [ ]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

<div class="alert alert-danger"><strong>STOP HERE AS NEEDED FOR CONCISE CSV TARGETS</strong></div>

# CELESTIAL COORDINATES FOR NEW STAR HD108959

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "HD 108959"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 8;  target_alt_name = "HD108959" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0IV'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR RASALHAGUE

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Rasalhague"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.07;  target_alt_name = "HD 159561" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR RASALGETHI

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Rasalgethi"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 5.3;  target_alt_name = "HD 156014" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR ALTAIR

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Altair"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 0.76;  target_alt_name = "HD 187642" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Vega

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Vega"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 0.026;  target_alt_name = "HD 172167" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Dubhe

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Dubhe"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 1.79;  target_alt_name = "HD 95689" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Scheat

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Scheat"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.42;  target_alt_name = "HD 217906" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Mizar

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Mizar"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.04;  target_alt_name = "HD 116656" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Alcor

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alcor"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 3.88;  target_alt_name = "HD 116657" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'MK'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR R Lyr

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "R Lyr"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 3.9;  target_alt_name = "HD 175865" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Alpheratz

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alpheratz"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.06;  target_alt_name = "HD 358" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8_A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Albireo A

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Albireo"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 3.21;  target_alt_name = "HD 183912" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Albireo B

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Albireo"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 5.11;  target_alt_name = "HD 183913" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Denebola

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Denebola"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.14;  target_alt_name = "HD 102647" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Zosma

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zosma"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.56;  target_alt_name = "HD 97603" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Alioth

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alioth"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 1.77;  target_alt_name = "HD 112185" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Minelauva

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Minelauva"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 3.32;  target_alt_name = "HD 112300" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M3'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Arcturus

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Arcturus"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = -0.05;  target_alt_name = "HD 124897" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR P Cyg

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "P Cyg"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 4.82;  target_alt_name = "HD 193237" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Polaris

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Polaris"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 1.98;  target_alt_name = "HD 8890" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Zet1 Lyr

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zet1 Lyr"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 4.37;  target_alt_name = "HD 173648" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'kA5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Zet2 Lyr

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zet2 Lyr"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 5.74;  target_alt_name = "HD 173649" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Kochab

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Kochab"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.08;  target_alt_name = "HD 131873" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Dschubba

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Dschubba"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 1.59;  target_alt_name = "HD 143275" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Enif

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Enif"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.37;  target_alt_name = "HD 206778" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Alphecca

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alphecca"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.24;  target_alt_name = "HD 139006" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Eltanin

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Eltanin"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 2.23;  target_alt_name = "HD 164058" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Thuban

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Thuban"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 3.67;  target_alt_name = "HD 123299" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR h Uma

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "h Uma"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 3.65;  target_alt_name = "HD 81937" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Theta Cep

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Theta Cep"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 4.22;  target_alt_name = "HD 195725" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR VZ Cam

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "VZ Cam"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 4.92;  target_alt_name = "HD 55966" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR Erakis

In [ ]:
from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Erakis"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 4.08;  target_alt_name = "HD 206936" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR 42 Her

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "42 Her"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 4.86;  target_alt_name = "HD 150450" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR NEW STAR V906 Her

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "V906 Her"
md(f"### Obtain spectral details for new star {target_name}")


In [ ]:
new_coord = obtain_info_for_star(target_name)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 6.60;  target_alt_name = "HD 150409" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Ma'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR Planet Neptune

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Neptune"
md(f"### Obtain spectral details for Planet {target_name}")


In [ ]:
#current 08/15/25 6:30 PM Pacific coordinates for Neptune
planet_ra = 0.079 #decimal deg
planet_dec = -1.344 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 7.8;  target_alt_name = "Neptune" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR Planet Saturn

In [ ]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Saturn"
md(f"### Obtain spectral details for Planet {target_name}")


In [ ]:
#current 08/15/25 6:30 PM Pacific coordinates for Neptune
planet_ra = 1.6195 #decimal deg
planet_dec = -1.9441 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 0.99;  target_alt_name = "Saturn" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# CELESTIAL COORDINATES FOR Planet Uranus

In [ ]:
# CELESTIAL COORDINATES FOR Planet Uranus

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Uranus"
md(f"### Obtain spectral details for Planet {target_name}")


In [ ]:
#current 08/15/25 6:30 PM Pacific coordinates for Uranus
planet_ra = 52.504 #decimal deg
planet_dec = 18.711 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 5.75;  target_alt_name = target_name # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# OUTPUT CONSOLIDATED CSV FILE FOR BARO

In [ ]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

# STOP HERE ADHOC CELESTIAL COORDINATES FOR ADHOC TARGET

In [ ]:
import ipywidgets as widgets
from IPython.display import display

text_input = widgets.Text(description='Enter Adhoc Target Name:')
output = widgets.Output()
current_text_value = ""

def on_text_change(change):
    global current_text_value
    current_text_value = change['new']
    with output:
        output.clear_output(wait=True)
        print(f"Current input value: {current_text_value}")

text_input.observe(on_text_change, names='value')
display(text_input, output)

# Create an float input widget for RA
ra_widget = widgets.FloatText(
    value=0.0000,
    description='Enter a RA in decimal degrees for adhoc target:',
    disabled=False
)

# Create an float input widget for DEC
dec_widget = widgets.FloatText(
    value=0.0000,
    description='Enter a DEC in decimal degrees for adhoc target:',
    disabled=False
)

display(ra_widget)
display(dec_widget)

# You can access the value in another cell or later in the same cell's execution
# with `num_widget.value`
# Example:
# my_number = num_widget.value
# print(f"The number entered is: {my_number}")


In [ ]:
my_tgt = text_input.value
my_ra = ra_widget.value
my_dec = dec_widget.value
print(f"The values entered are: {my_tgt} & {my_ra} & {my_dec}")

In [ ]:
new_coord = obtain_info_for_adhoc_star(my_tgt, my_ra, my_dec)

### Update Properties for Star based on wiki query results above 

In [ ]:
star_magnitude = 12.9;  target_alt_name = "Quasar_3C273" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

In [ ]:
star_magnitude = 12.9;  target_alt_name = my_tgt # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

In [ ]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])